# Praktikum 4 — Perbandingan Annoy, FAISS, dan HNSW

## Tujuan
Membandingkan waktu pembangunan indeks (*build time*) dan waktu pencarian (*query time*) dari tiga pendekatan ANN/NN:
- **Annoy** — random projection trees
- **FAISS `IndexFlatL2`** — pencarian exact dengan L2
- **HNSW (`hnswlib`)** — pencarian berbasis graf

Eksperimen awal menggunakan **1.000.000 data, dimensi 5, dan k = 10**, seperti instruksi praktikum. Notebook juga memungkinkan variasi metrik **Euclidean/L2, Cosine, dan Inner Product**.

> **Penting:** `IndexFlatL2` di FAISS hanya mendukung pencarian L2 dalam konfigurasi yang digunakan di sini. Untuk perbandingan Cosine dan Inner Product, notebook menggunakan FAISS `IndexFlatIP` (cosine dilakukan dengan normalisasi vektor) serta metrik terkait pada Annoy dan HNSW. Jadi, nama indeks FAISS berubah sesuai metrik.
>
> Waktu build dan query akan diukur terpisah. Angka hasil bergantung pada runtime Google Colab dan harus dihasilkan dengan menjalankan notebook; tidak ada hasil waktu yang direkayasa atau diisi secara manual.


In [1]:
# Instalasi library di Google Colab
!pip -q install annoy faiss-cpu hnswlib pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.0 MB/s eta 0:00:00


In [2]:
import gc
import time
import numpy as np
import pandas as pd
from IPython.display import display

from annoy import AnnoyIndex
import faiss
import hnswlib

SEED = 42
np.random.seed(SEED)

# Pengaturan eksperimen
N_DATA = 1_000_000       # ubah menjadi 100_000 jika RAM terbatas
DIM = 5
K = 10
ANNOY_TREES = 10
HNSW_M = 16
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 50

# True menjalankan semua metrik; False menjalankan hanya Euclidean/L2.
RUN_ALL_METRICS = True

print("Konfigurasi:", {
    "n_data": N_DATA, "dim": DIM, "k": K,
    "Annoy trees": ANNOY_TREES,
    "HNSW M": HNSW_M,
    "HNSW ef_construction": HNSW_EF_CONSTRUCTION,
    "HNSW ef_search": HNSW_EF_SEARCH
})


Konfigurasi: {'n_data': 1000000, 'dim': 5, 'k': 10, 'Annoy trees': 10, 'HNSW M': 16, 'HNSW ef_construction': 200, 'HNSW ef_search': 50}


## Fungsi eksperimen

Untuk membuat perbandingan yang adil, dataset dan query yang sama dipakai untuk semua library dalam satu metrik. Waktu pemasangan library dan pembuatan dataset tidak dihitung sebagai waktu build indeks. Waktu build mencakup penambahan data dan proses `build()` pada Annoy, `add()` pada FAISS, serta inisialisasi dan penambahan data pada HNSW.

In [3]:
def normalize_rows(A):
    norms = np.linalg.norm(A, axis=1, keepdims=True)
    return A / np.maximum(norms, 1e-12)


def make_metric_data(X, query, metric):
    """Kembalikan data/query dan konfigurasi yang sesuai untuk tiap metrik."""
    if metric == "euclidean":
        return X, query, "euclidean", "l2", "IndexFlatL2"
    if metric == "cosine":
        # Annoy angular dan HNSW cosine mendukung pencarian berbasis sudut.
        # FAISS IP setelah normalisasi setara dengan cosine similarity.
        return normalize_rows(X), normalize_rows(query), "angular", "cosine", "IndexFlatIP (normalized)"
    if metric == "inner_product":
        # Inner product dimaksimalkan. Annoy tidak menyediakan IP langsung,
        # jadi untuk perbandingan tiga library ini metrik IP hanya dijalankan FAISS/HNSW.
        return X, query, None, "ip", "IndexFlatIP"
    raise ValueError("Metrik tidak dikenal")


def run_one_metric(X, query, metric, k=K):
    """Jalankan pembandingan library untuk satu metrik; hasil disimpan sebagai dict."""
    X_metric, query_metric, annoy_metric, hnsw_space, faiss_name = make_metric_data(X, query, metric)
    rows = []

    # ---- Annoy ----
    if annoy_metric is not None:
        ann = AnnoyIndex(X.shape[1], annoy_metric)
        t0 = time.perf_counter()
        for i, vec in enumerate(X_metric):
            ann.add_item(i, vec.tolist())
        ann.build(ANNOY_TREES)
        annoy_build = time.perf_counter() - t0

        t0 = time.perf_counter()
        ann_ids, ann_dist = ann.get_nns_by_vector(
            query_metric[0].tolist(), k, include_distances=True
        )
        annoy_query = time.perf_counter() - t0

        rows.append({
            "Metrik": metric, "Library/Index": f"Annoy ({annoy_metric})",
            "Build time (s)": annoy_build, "Query time (s)": annoy_query,
            "Top-5/10 indices": ann_ids[:5], "Top-k indices": ann_ids,
            "Top-k distances/scores": ann_dist,
            "Catatan": "Approximate"
        })
        del ann

    # ---- FAISS ----
    if metric == "euclidean":
        faiss_index = faiss.IndexFlatL2(X.shape[1])
        query_for_faiss = query_metric.astype(np.float32)
        t0 = time.perf_counter()
        faiss_index.add(X_metric)
        faiss_build = time.perf_counter() - t0
        t0 = time.perf_counter()
        D, I = faiss_index.search(query_for_faiss, k)
        faiss_query = time.perf_counter() - t0
        distances_or_scores = D[0].tolist()
        note = "Exact; FAISS mengembalikan squared L2 distance"
    elif metric == "cosine":
        faiss_index = faiss.IndexFlatIP(X.shape[1])
        query_for_faiss = query_metric.astype(np.float32)
        t0 = time.perf_counter()
        faiss_index.add(X_metric.astype(np.float32))
        faiss_build = time.perf_counter() - t0
        t0 = time.perf_counter()
        D, I = faiss_index.search(query_for_faiss, k)
        faiss_query = time.perf_counter() - t0
        distances_or_scores = D[0].tolist()
        note = "Exact; inner product vektor ternormalisasi = cosine similarity"
    else:
        faiss_index = faiss.IndexFlatIP(X.shape[1])
        query_for_faiss = query_metric.astype(np.float32)
        t0 = time.perf_counter()
        faiss_index.add(X_metric)
        faiss_build = time.perf_counter() - t0
        t0 = time.perf_counter()
        D, I = faiss_index.search(query_for_faiss, k)
        faiss_query = time.perf_counter() - t0
        distances_or_scores = D[0].tolist()
        note = "Exact; inner product terbesar adalah yang terbaik"

    rows.append({
        "Metrik": metric, "Library/Index": f"FAISS ({faiss_name})",
        "Build time (s)": faiss_build, "Query time (s)": faiss_query,
        "Top-5/10 indices": I[0][:5].astype(int).tolist(),
        "Top-k indices": I[0].astype(int).tolist(),
        "Top-k distances/scores": distances_or_scores,
        "Catatan": note
    })
    del faiss_index

    # ---- HNSW ----
    hnsw = hnswlib.Index(space=hnsw_space, dim=X.shape[1])
    t0 = time.perf_counter()
    hnsw.init_index(
        max_elements=X.shape[0],
        ef_construction=HNSW_EF_CONSTRUCTION,
        M=HNSW_M,
        random_seed=SEED
    )
    hnsw.add_items(X_metric, np.arange(X.shape[0]))
    hnsw_build = time.perf_counter() - t0
    hnsw.set_ef(max(HNSW_EF_SEARCH, k))

    t0 = time.perf_counter()
    labels, distances = hnsw.knn_query(query_metric, k=k)
    hnsw_query = time.perf_counter() - t0

    rows.append({
        "Metrik": metric, "Library/Index": f"HNSW ({hnsw_space})",
        "Build time (s)": hnsw_build, "Query time (s)": hnsw_query,
        "Top-5/10 indices": labels[0][:5].astype(int).tolist(),
        "Top-k indices": labels[0].astype(int).tolist(),
        "Top-k distances/scores": distances[0].tolist(),
        "Catatan": "Approximate; jarak L2 dari hnswlib berupa squared L2"
    })
    del hnsw
    gc.collect()
    return rows


## Jalankan eksperimen

Dataset dibuat satu kali lalu digunakan kembali untuk semua metrik. Untuk Euclidean, ketiga library dibandingkan. Untuk Cosine, Annoy Angular, FAISS inner product pada vektor ternormalisasi, dan HNSW Cosine digunakan. Untuk Inner Product, FAISS dan HNSW dibandingkan karena Annoy tidak menyediakan metrik inner product secara langsung.

In [4]:
rng = np.random.default_rng(SEED)
X = rng.random((N_DATA, DIM), dtype=np.float32)
query = rng.random((1, DIM), dtype=np.float32)

metrics_to_run = ["euclidean", "cosine", "inner_product"] if RUN_ALL_METRICS else ["euclidean"]
all_rows = []

for metric in metrics_to_run:
    print(f"Menjalankan metrik: {metric} ...")
    try:
        metric_rows = run_one_metric(X, query, metric, k=K)
        all_rows.extend(metric_rows)
        for row in metric_rows:
            print(
                f"  {row['Library/Index']}: build={row['Build time (s)']:.3f}s, "
                f"query={row['Query time (s)']:.6f}s"
            )
    except Exception as exc:
        print(f"  Gagal pada metrik {metric}: {exc}")
        print("  Jika RAM terbatas, ubah N_DATA menjadi 100_000 lalu jalankan ulang.")
    gc.collect()

results_df = pd.DataFrame(all_rows)
display(results_df[[
    "Metrik", "Library/Index", "Build time (s)", "Query time (s)",
    "Top-5/10 indices", "Catatan"
]] if not results_df.empty else results_df)


Menjalankan metrik: euclidean ...
  Annoy (euclidean): build=14.124s, query=0.000096s
  FAISS (IndexFlatL2): build=0.006s, query=0.006510s
  HNSW (l2): build=110.536s, query=0.000087s
Menjalankan metrik: cosine ...
  Annoy (angular): build=16.281s, query=0.000088s
  FAISS (IndexFlatIP (normalized)): build=0.009s, query=0.002400s
  HNSW (cosine): build=103.227s, query=0.000072s
Menjalankan metrik: inner_product ...
  FAISS (IndexFlatIP): build=0.006s, query=0.002404s
  HNSW (ip): build=65.745s, query=0.000076s


,Metrik,Library/Index,Build time (s),Query time (s),Top-5/10 indices,Catatan
0,euclidean,Annoy (euclidean),14.123999,0.000096,"[85158, 586716, 687018, 8630, 91763]",Approximate
1,euclidean,FAISS (IndexFlatL2),0.005672,0.006510,"[85158, 586716, 687018, 8630, 91763]",Exact; FAISS mengembalikan squared L2 distance
2,euclidean,HNSW (l2),110.535568,0.000087,"[85158, 586716, 687018, 8630, 91763]",Approximate; jarak L2 dari hnswlib berupa squa...
3,cosine,Annoy (angular),16.281145,0.000088,"[233781, 94411, 687018, 914994, 906067]",Approximate
4,cosine,FAISS (IndexFlatIP (normalized)),0.008892,0.002400,"[233781, 94411, 687018, 914994, 906067]",Exact; inner product vektor ternormalisasi = c...
5,cosine,HNSW (cosine),103.227052,0.000072,"[233781, 94411, 687018, 914994, 906067]",Approximate; jarak L2 dari hnswlib berupa squa...
6,inner_product,FAISS (IndexFlatIP),0.006410,0.002404,"[464222, 991290, 208679, 369761, 649453]",Exact; inner product terbesar adalah yang terbaik
7,inner_product,HNSW (ip),65.744753,0.000076,"[464222, 991290, 208679, 369761, 649453]",Approximate; jarak L2 dari hnswlib berupa squa...


## Tabel laporan praktikum

Gunakan tabel berikut untuk mencatat hasil aktual dari runtime. Indeks tetangga yang diperoleh library approximate dapat berbeda dari FAISS Flat karena strategi pencariannya berbeda. Perbandingan indeks bermakna langsung terutama untuk metrik yang sama.

In [5]:
if not results_df.empty:
    report_df = results_df[[
        "Metrik", "Library/Index", "Top-k indices",
        "Build time (s)", "Query time (s)", "Top-k distances/scores", "Catatan"
    ]].copy()
    display(report_df)

    csv_path = "hasil_praktikum_4_annoy_faiss_hnsw.csv"
    report_df.to_csv(csv_path, index=False)
    print("Hasil tersimpan ke:", csv_path)
else:
    print("Belum ada hasil. Jalankan sel eksperimen terlebih dahulu.")


,Metrik,Library/Index,Top-k indices,Build time (s),Query time (s),Top-k distances/scores,Catatan
0,euclidean,Annoy (euclidean),"[85158, 586716, 687018, 8630, 91763, 4003, 494...",14.123999,0.000096,"[0.04701298847794533, 0.05091444030404091, 0.0...",Approximate
1,euclidean,FAISS (IndexFlatL2),"[85158, 586716, 687018, 8630, 91763, 4003, 494...",0.005672,0.006510,"[0.002210378646850586, 0.002592325210571289, 0...",Exact; FAISS mengembalikan squared L2 distance
2,euclidean,HNSW (l2),"[85158, 586716, 687018, 8630, 91763, 4003, 494...",110.535568,0.000087,"[0.0022102207876741886, 0.002592280274257064, ...",Approximate; jarak L2 dari hnswlib berupa squa...
3,cosine,Annoy (angular),"[233781, 94411, 687018, 914994, 906067, 4003, ...",16.281145,0.000088,"[0.02608308754861355, 0.02861969918012619, 0.0...",Approximate
4,cosine,FAISS (IndexFlatIP (normalized)),"[233781, 94411, 687018, 914994, 906067, 4003, ...",0.008892,0.002400,"[0.9996598362922668, 0.9995903372764587, 0.999...",Exact; inner product vektor ternormalisasi = c...
5,cosine,HNSW (cosine),"[233781, 94411, 687018, 914994, 906067, 4003, ...",103.227052,0.000072,"[0.0003400444984436035, 0.0004093647003173828,...",Approximate; jarak L2 dari hnswlib berupa squa...
6,inner_product,FAISS (IndexFlatIP),"[464222, 991290, 208679, 369761, 649453, 69012...",0.006410,0.002404,"[2.18001127243042, 2.162381887435913, 2.161738...",Exact; inner product terbesar adalah yang terbaik
7,inner_product,HNSW (ip),"[464222, 991290, 208679, 369761, 649453, 69012...",65.744753,0.000076,"[-1.18001127243042, -1.162381887435913, -1.161...",Approximate; jarak L2 dari hnswlib berupa squa...


Hasil tersimpan ke: hasil_praktikum_4_annoy_faiss_hnsw.csv


## Kesimpulan yang dapat diisi setelah eksperimen

- **Waktu build:** bandingkan biaya pembuatan indeks masing-masing library.
- **Waktu query:** bandingkan waktu pencarian setelah indeks siap.
- **Akurasi:** FAISS Flat memberikan hasil exact; Annoy dan HNSW adalah approximate pada konfigurasi ini.
- **Metrik:** Annoy mendukung Euclidean dan Angular; FAISS Flat mendukung L2 dan inner product; cosine dapat dilakukan melalui normalisasi lalu inner product. HNSW mendukung L2, cosine, dan inner product.
- **Keterbatasan:** satu kali pengukuran bisa dipengaruhi warm-up dan kondisi runtime. Untuk laporan yang lebih kuat, jalankan beberapa kali dan gunakan median waktu.
- **Catatan memori:** dataset 1 juta × 5 menggunakan sekitar 20 MB untuk array float32, tetapi struktur indeks dan salinan internal membutuhkan memori tambahan. Jika runtime terhenti, turunkan `N_DATA` ke `100_000`.
